# Self-Validating Bayesian Optimization for Cu₃VS₄ Nanoparticle Synthesis

This notebook is the **execution interface** — all logic lives in Python modules under `src/`.

| Module | What it does |
|---|---|
| `config.py` | Tunable parameters (bounds, feature toggles, plot style) |
| `features.py` | Feature engineering and smart hybrid selection |
| `optimizer.py` | Base `Cu3VS4Optimizer` with GP models |
| `selfvalidating.py` | Self-validation, error learning, bias correction |
| `experiment_store.py` | Experiment & recommendation JSON persistence |
| `diagnostics.py` | LOO-CV, VIF, calibration, convergence metrics |
| `visualization.py` | All plotting functions |
| `chemical_constants.py` | Physical/chemical descriptors |

**Workflow:** Setup → Initialize → Recommend → Complete → Assess → Visualize

In [ ]:
# =============================================================================
# 0. IMPORTS & SETUP
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

import sys
sys.path.insert(0, str(Path("..").resolve() / "src"))

# Core
from selfvalidating import SelfValidatingOptimizer
from config import COLORS, PUBLICATION_STYLE

# Visualization
from visualization import (
    plot_recommendation_history, plot_parity, plot_calibration,
    plot_error_learning_progress, plot_error_correction_impact,
    plot_target_achievement, plot_feature_importance,
    plot_classifier_calibration, plot_collinearity_heatmap,
    plot_dataset_quality_dashboard, plot_loo_residuals,
    plot_property_correlations, plot_acquisition_slice,
    plot_recommendation_regret,
)

# Diagnostics & reporting
from diagnostics import (
    print_model_assessment, display_recommendations_table,
    print_optimization_convergence_summary,
)

# Paths
DATA_DIR = Path("..") / "data"
OUTPUT_DIR = Path("..") / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Style
plt.rcParams.update(PUBLICATION_STYLE)
warnings.filterwarnings('ignore')
np.random.seed(42)

print("All imports successful!")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

---
## 1. Initialize the Optimizer

Loads experiments from JSON (or imports from CSV on first run), builds GP
models, loads recommendation history, and initializes the error learner.

**Feature mode** can be changed in `config.py` or overridden below:
- `'smart_hybrid'` — *recommended* — auto-removes collinear features via VIF
- `'hybrid'` — raw parameters + key chemical ratios
- `'chemical'` — fully transformed chemical features
- `'raw'` — original synthesis parameters only

In [ ]:
optimizer = SelfValidatingOptimizer(
    data_dir=DATA_DIR,
    initialize_from_csv=True,
    feature_mode='smart_hybrid',
)

---
## 2. Generate Recommendations

Set your target particle size below and run to get optimized synthesis conditions.
Recommendations are automatically stored in `recommendations.json`.

In [ ]:
TARGET_SIZE = 16.0   # nm -- MODIFY THIS
SIZE_TOLERANCE = 2.0  # acceptable range: target +/- tolerance

recommendations = optimizer.recommend(
    target_size=TARGET_SIZE,
    size_tol=SIZE_TOLERANCE,
    seed=42,
)

display_recommendations_table(recommendations)
display(recommendations)

---
## 3. View Pending Recommendations

See what recommendations are waiting for experimental results.

In [ ]:
pending = optimizer.get_pending_recommendations()
display(pending)

---
## 4. Complete Recommendations (After Lab Work)

After running an experiment, uncomment the block below and fill in your
measured results. This logs the data, computes prediction errors, retrains
the models, and updates the error learner.

In [ ]:
# Uncomment and fill in when you have results:

# errors = optimizer.complete_recommendation(
#     rec_id='REC_003',       # <-- your recommendation ID
#     Size=16.25,              # measured size (nm)
#     GSD=1.58,                # measured GSD
#     Squareness=0.67,         # measured squareness
#     HasProduct=1,            # 1 = product formed
#     PhasePure=1,             # 1 = phase pure
#     Polymorph='cubic',       # 'cubic', 'tetragonal', etc.
# )

---
## 5. Model Assessment

### 5a. Self-Assessment Report

Summary of experiment database, recommendation history, prediction accuracy
(from completed recommendations), and error-correction status.

In [ ]:
print_model_assessment(optimizer)

### 5b. Comprehensive Diagnostics

LOO-CV metrics, collinearity (VIF), classifier calibration, sample-size adequacy.

In [ ]:
diagnostics = optimizer.full_diagnostics()

### 5c. Feature Mode Comparison *(optional, takes 2-3 min)*

Compares LOO-CV R-squared across `raw`, `chemical`, `hybrid`, and `smart_hybrid`
to verify the chosen mode is optimal for your data.

In [ ]:
# Uncomment to run:
# comparison_df = optimizer.compare_feature_modes(
#     modes=['raw', 'chemical', 'hybrid', 'smart_hybrid'],
#     verbose=True,
# )

### 5d. Convergence Check

Are we still improving, or has optimization plateaued?
Shows success rate (within tolerance), mean |error|, and per-size-band metrics.

In [ ]:
print_optimization_convergence_summary(optimizer, last_n=10, size_band_nm=5.0)

---
## 6. Visualization

### 6a. Self-Validation Plots

These show how well the model predicts experimental outcomes from completed
recommendations — the core value of the self-validating loop.

In [ ]:
# Recommendation timeline + error distribution
fig = plot_recommendation_history(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'recommendation_history.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Predicted vs Actual for completed recommendations
fig = plot_parity(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'parity_plots.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Confidence interval coverage (z-score distribution)
fig = plot_calibration(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'calibration_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Error correction impact: with vs without bias correction
fig = plot_error_correction_impact(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'error_correction_impact.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Target achievement: how close are we hitting the requested sizes?
fig = plot_target_achievement(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'target_achievement.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Error learning progress: is cumulative MAE trending down?
fig = plot_error_learning_progress(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'learning_progress.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Simple regret over completed recommendations
fig = plot_recommendation_regret(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'recommendation_regret.png', dpi=300, bbox_inches='tight')
    plt.show()

### 6b. Model Quality Plots

Assess the GP models themselves: fit quality, feature importance, and
multicollinearity diagnostics.

In [ ]:
# LOO-CV parity and residuals (3x2 grid)
fig = plot_loo_residuals(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'loo_residuals.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Feature importance from GP lengthscales
fig = plot_feature_importance(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Classifier reliability diagrams (HasProduct, PhasePure, IsCubic)
fig = plot_classifier_calibration(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'classifier_calibration.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Feature correlation heatmap
fig = plot_collinearity_heatmap(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'feature_correlations.png', dpi=300, bbox_inches='tight')
    plt.show()

### 6c. Data Exploration

Visualize the dataset itself: property trade-offs, design-space coverage,
and where the acquisition function is searching.

In [ ]:
# Property correlations: Size vs GSD vs Squareness
fig = plot_property_correlations(optimizer)
if fig:
    plt.savefig(OUTPUT_DIR / 'property_correlations.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# 2D acquisition slice (Temp vs log_Time, others at median)
fig = plot_acquisition_slice(optimizer, target_size=TARGET_SIZE, size_tol=SIZE_TOLERANCE)
if fig:
    plt.savefig(OUTPUT_DIR / 'acquisition_slice.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Comprehensive 4-panel dashboard (good for presentations)
fig = plot_dataset_quality_dashboard(
    optimizer, figsize=(14, 10),
    save_path=OUTPUT_DIR / 'dataset_quality_dashboard.png',
)
if fig:
    plt.show()

---
## 7. Utilities

### Add a Manual Experiment

Add experiments you designed yourself (not from a recommendation).

In [ ]:
# Uncomment and fill in:

# exp_id = optimizer.add_manual_experiment(
#     Temp=285.0, Time=30.0, VOacac=0.25, DDT=3.0, OAm=4.0,
#     Size=19.5, GSD=1.30, Squareness=0.62,
#     HasProduct=1, PhasePure=1, Polymorph='cubic',
# )
# print(f"Added: {exp_id}")

### Test Predictions

Predict outcomes for specific conditions without generating a recommendation.

In [ ]:
result = optimizer.predict_from_conditions(
    Temp=285.0, Time=30.0, VOacac=0.20, DDT=3.0, OAm=4.5,
    verbose=True,
)

### Inspect GP Kernels

View the fitted kernel parameters (lengthscales, noise, signal variance).

In [ ]:
base = optimizer.base_optimizer
for name, gp in [("Size", base.gp_size), ("GSD", base.gp_gsd), ("Squareness", base.gp_sq)]:
    print(f"\n{name} fitted kernel:")
    print(f"  {gp.kernel_}")
    print(f"  Log-marginal-likelihood: {gp.log_marginal_likelihood_value_:.3f}")